In [ ]:
# Cell: Sync latest code from GitHub
!cd /content/SentimentAnalysis && git pull origin main

/bin/bash: line 1: cd: /content/SentimentAnalysis: No such file or directory


In [ ]:
%cd /content
!pwd

/content
/content


In [ ]:
!rm -rf /content/SentimentAnalysis
!git clone https://github.com/Chetnapadhi/SentimentAnalysis.git /content/SentimentAnalysis

Cloning into '/content/SentimentAnalysis'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 51 (delta 7), reused 48 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 80.56 KiB | 11.51 MiB/s, done.
Resolving deltas: 100% (7/7), done.


In [ ]:
%cd /content/SentimentAnalysis
!pwd
!ls

/content/SentimentAnalysis
/content/SentimentAnalysis
config.yaml  models	README.md	  run_e0.py  smoke_test_e0.py
data	     notebooks	requirements.txt  run_e1.py  src


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cpu
CUDA available: False


In [ ]:
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 106.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!mkdir -p /content/drive/MyDrive/SentimentAnalysis/results/E1

In [ ]:
import os

canonical_dir = "/content/SentimentAnalysis/data/processed/canonical"

for f in [
    "final_train.jsonl",
    "final_validation.jsonl",
    "final_test.jsonl"
]:
    path = os.path.join(canonical_dir, f)
    print(f"{f}: {'FOUND' if os.path.exists(path) else 'MISSING'}")

final_train.jsonl: MISSING
final_validation.jsonl: MISSING
final_test.jsonl: MISSING


In [ ]:

!python -m src.data.inspect_datasets

README.md: 100% 1.39k/1.39k [00:00<00:00, 1.41MB/s]
train-emoji-bear-unmodified.txt: 100% 561k/561k [00:00<00:00, 81.9MB/s]

train-emoji-bull-unmodified.txt: downloading bytes:   0% 0.00/4.56M [00:00<?, ?B/s]
train-emoji-bull-unmodified.txt: downloading bytes: 100% 2.96M/2.96M [00:01<00:00, 1.80MB/s,  286kB/s  ]
train-emoji-bull-unmodified.txt: reconstructing file: 100% 4.56M/4.56M [00:01<00:00, 2.77MB/s,  440kB/s  ]

train-emoji-net-unmodified.txt: downloading bytes:  15% 322k/2.13M [00:01<00:08, 217kB/s]
train-emoji-net-unmodified.txt: downloading bytes: 100% 1.42M/1.42M [00:01<00:00, 944kB/s,  138kB/s  ]
train-emoji-net-unmodified.txt: reconstructing file: 100% 2.13M/2.13M [00:01<00:00, 1.41MB/s,  206kB/s  ]

train-svm-bear-neg.txt: downloading bytes:   0% 0.00/1.58M [00:00<?, ?B/s]
train-svm-bear-neg.txt: downloading bytes: 100% 1.08M/1.08M [00:01<00:00, 782kB/s,  105kB/s  ]
train-svm-bear-neg.txt: reconstructing file: 100% 1.58M/1.58M [00:01<00:00, 1.15MB/s,  153kB/s  ]

train-svm

In [ ]:
!python -m src.data.stocktwits_adapter

Creating CSV from Arrow format: 100% 211/211 [00:01<00:00, 115.87ba/s]
Saved train: 210699 rows -> data/processed/stocktwits_train.csv
Creating CSV from Arrow format: 100% 21/21 [00:00<00:00, 114.03ba/s]
Saved validation: 20676 rows -> data/processed/stocktwits_validation.csv
Creating CSV from Arrow format: 100% 12/12 [00:00<00:00, 112.76ba/s]
Saved test: 11966 rows -> data/processed/stocktwits_test.csv
Report saved -> data/inspection/stocktwits_labels.json

STOCKTWITS LABEL RECOVERY COMPLETE

TRAIN:
  HF rows:       211758
  Matched:       210699
  Dropped:       1059 (conflicting labels)
  Class dist:    {'Bearish': 35843, 'Neutral': 63593, 'Bullish': 111263}
  Emoji records: 210699

VALIDATION:
  HF rows:       20761
  Matched:       20676
  Dropped:       85 (conflicting labels)
  Class dist:    {'Bearish': 4073, 'Neutral': 7496, 'Bullish': 9107}
  Emoji records: 20676

TEST:
  HF rows:       11984
  Matched:       11966
  Dropped:       18 (conflicting labels)
  Class dist:    {'B

In [ ]:
!python -m src.data.build_final_dataset

Saved raw snapshots: stocktwits_{train,validation,test}_raw.csv

FINAL DATASET CONSTRUCTION REPORT
Training deduplication:
  HF train rows:              211758
  After conflict-label drop:   210699 (adapter output)
  Duplicate rows removed:      119551
  Train<->test overlap removed:27
  Final training rows:         91121
  Validation (unchanged):      20676
  Test (unchanged):            11966

Class distribution (train original -> final):
  Bearish: 35843 -> 7290
  Neutral: 63593 -> 26237
  Bullish: 111263 -> 57594

Class weights (train-only, final):
  Bearish: 4.1665
  Neutral: 1.1577
  Bullish: 0.5274

Manifest written -> data/processed/experiment_manifest.json


In [ ]:
!python -m src.data.preprocessing

Saved 91121 canonical rows -> data/processed/canonical/final_train.jsonl
Saved 20676 canonical rows -> data/processed/canonical/final_validation.jsonl
Saved 11966 canonical rows -> data/processed/canonical/final_test.jsonl


In [ ]:
!ls -lh data/processed/canonical/

total 39M
-rw-r--r-- 1 root root 4.0M Sep  7 10:43 final_test.jsonl
-rw-r--r-- 1 root root  28M Sep  7 10:43 final_train.jsonl
-rw-r--r-- 1 root root 6.7M Sep  7 10:43 final_validation.jsonl


In [ ]:
import os
os.environ["PYTHONPATH"] = "/content/SentimentAnalysis"

In [ ]:
%cd /content/SentimentAnalysis

import os
os.environ["RUN_E1"] = "1"

# Import the E1 training module
import src.train_e1 as e1

print("RUN_E1 =", os.environ.get("RUN_E1"))
print("train_e1 exists:", hasattr(e1, "train_e1"))

In [ ]:
!find results/E1 -maxdepth 2 -type f | sort

In [ ]:
import json

with open("results/E1/metrics.json", "r") as f:
    metrics = json.load(f)

print(json.dumps(metrics, indent=2))

In [ ]:
print(open("results/E1/E1_report.md", "r", encoding="utf-8").read())

In [ ]:
!cp -r results/E1/* /content/drive/MyDrive/SentimentAnalysis/results/E1/

In [ ]:
!find /content/drive/MyDrive/SentimentAnalysis/results/E1 -maxdepth 2 -type f | sort